# RG3 비라벨 선행 공정관계 + 3모델 앙상블
평가기간보다 앞선 비라벨만 참조. 비라벨을 정상으로 가정하지 않음.

In [1]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix,average_precision_score
from catboost import CatBoostClassifier
from sklearn.base import clone
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
limit=d.TimeStamp.min()-pd.Timedelta(hours=1)
u=u.loc[u.EQUIP_CD.eq('S14')&u.TimeStamp.lt(limit)].copy()
assert u.TimeStamp.max()<limit
u=u.loc[~(u.Barrel_Temperature_1.eq(0)&u.Max_Screw_RPM.eq(0)&u.Average_Screw_RPM.eq(0))]
specs={
'pressure':('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time','Average_Back_Pressure','Barrel_Temperature_2','Barrel_Temperature_5','Mold_Temperature_3']),
'cushion':('Cushion_Position',['Plasticizing_Position','Max_Switch_Over_Pressure','Average_Back_Pressure','Plasticizing_Time']),
'thermal_gradient':('thermal_gradient',['Hopper_Temperature','Mold_Temperature_3','Plasticizing_Time','Average_Back_Pressure']),
'fill_gap':('fill_gap',['Max_Injection_Speed','Max_Injection_Pressure','Average_Back_Pressure'])}
for frame in [u,d]:
    frame['thermal_gradient']=frame.Barrel_Temperature_2-frame.Barrel_Temperature_5
    frame['fill_gap']=frame.Injection_Time-frame.Filling_Time
allcols=sorted(set([t for t,p in specs.values()]+sum([p for t,p in specs.values()],[])))
keys=['EQUIP_CD','TimeStamp']
conflicts=(u.groupby(keys)[allcols].max()-u.groupby(keys)[allcols].min()).gt(1e-4).any(axis=1)
bad=conflicts[conflicts].index
u=u.loc[~pd.MultiIndex.from_frame(u[keys]).isin(bad)].sort_values('TimeStamp').drop_duplicates(keys)
if len(u)>12000:u=u.iloc[np.linspace(0,len(u)-1,12000).astype(int)].copy()
u=u.reset_index(drop=True)
split=int(len(u)*.8);fit=u.iloc[:split];cal=u.iloc[split:]
assert len(fit)>500 and len(cal)>100
new=[];reference_audit=[]
for name,(target,predictors) in specs.items():
    good=fit[target].notna()
    reg=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),HistGradientBoostingRegressor(loss='absolute_error',max_iter=150,max_leaf_nodes=7,min_samples_leaf=30,l2_regularization=10,random_state=42))
    reg.fit(fit.loc[good,predictors],fit.loc[good,target])
    residual=cal[target].to_numpy()-reg.predict(cal[predictors])
    center=float(np.nanmedian(residual))
    floor=1. if name=='pressure' else .01 if name in ['cushion','fill_gap'] else .1
    scale=max(float(np.nanmedian(np.abs(residual-center)))*1.4826,floor)
    new.append((d[target].to_numpy()-reg.predict(d[predictors])-center)/scale)
    reference_audit.append(dict(relation=name,fit_shots=len(fit),cal_shots=len(cal),pool_start=u.TimeStamp.min(),pool_end=u.TimeStamp.max(),label_start=d.TimeStamp.min(),scale=scale,center=center,predictors='|'.join(predictors)))
    print('External relation fitted',name,flush=True)
R=np.column_stack(new)
pd.DataFrame(reference_audit).to_csv(tables/'rg3_unlabeled_relation_reference.csv',index=False,encoding='utf-8-sig')
ft=pd.DataFrame(R,columns=[name+'_external_residual_z' for name in specs])
ft.insert(0,'TimeStamp',d.TimeStamp);ft['actual']=y
ft.to_csv(tables/'rg3_unlabeled_relation_features.csv',index=False,encoding='utf-8-sig')
A=np.column_stack([d[stdcols].to_numpy(float),R])
models={'augmented_RF':RandomForestClassifier(n_estimators=200,max_depth=6,min_samples_leaf=5,class_weight='balanced_subsample',random_state=42,n_jobs=-1),
'augmented_ET':ExtraTreesClassifier(n_estimators=200,max_depth=6,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1),
'augmented_Cat':CatBoostClassifier(iterations=200,depth=3,learning_rate=.03,l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)}
modelcache={}
def predict_aug(name,ref,q):
    key=(name,tuple(ref),tuple(q))
    if key in modelcache:return modelcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:score=np.full(len(q),float(y[ref][0]) if len(ref) else 0.)
    else:
        model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),clone(models[name]))
        score=model.fit(A[ref],y[ref]).predict_proba(A[q])[:,1]
    modelcache[key]=score
    return score
records=[];folds=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=np.column_stack([predict_aug(name,ref,q) for name in models])
        inner=np.full((len(ref),3),np.nan);old_inner=np.full(len(ref),np.nan)
        old_outer=pair_predict(ref,q).max(axis=1)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                for k,name in enumerate(models):inner[mask,k]=predict_aug(name,ref[~mask],ref[mask])
                old_inner[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
        candidates={name:(outer[:,k],inner[:,k]) for k,name in enumerate(models)}
        candidates['augmented3_mean']=(outer.mean(axis=1),inner.mean(axis=1))
        candidates['old_augmented_mean']=((old_outer+outer.mean(axis=1))/2,(old_inner+inner.mean(axis=1))/2)
        candidates['old_augmented_max']=(np.maximum(old_outer,outer.mean(axis=1)),np.maximum(old_inner,inner.mean(axis=1)))
        candidates['RF_baseline']=(old_outer,old_inner)
        for name,(score,inside) in candidates.items():
            normal=inside[(y[ref]==0)&np.isfinite(inside)]
            for rate in [.005,.01,.02]:
                cut=float(np.quantile(normal,1-rate,method='higher')) if len(normal) else 1.
                for k,z in enumerate(q):records.append(dict(scheme=scheme,model=name,normal_budget=rate,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[k]),threshold=cut,prediction=int(score[k]>cut)))
        folds.append(dict(scheme=scheme,block=block,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum())))
        print('Completed unlabeled reference',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name,budget),g in pp.groupby(['scheme','model','normal_budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
    rows.append(dict(scheme=scheme,model=name,normal_budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=f1))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_unlabeled_relation_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_unlabeled_relation_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(folds).to_csv(tables/'rg3_unlabeled_relation_folds.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.normal_budget.eq(.01)].sort_values(['scheme','F1','AP'],ascending=[True,False,False])
lines=['# RG3 비라벨 공정참조 관계 피처 및 앙상블','',
'센서관계참조는S14/RG3비라벨샷중가장빠른라벨시각보다1시간이상앞선자료만사용한다. 라벨기간또는미래생산자료를참조학습에포함하지않았다. 모든정답을사용해정상으로선별하지않음.',
'배럴1및RPM2개가동시에0인비가동모드를제외하고동일설비/시각중복을제거했다. 중복센서불일치샷제외. 시간순최대12000샷샘플,앞80%관계회귀/뒤20%오차MAD척도산출.',
'비라벨정상보장은없다. absolute_error 비선형회귀로전형적인관계참조를학습할뿐품질정상모델이아니다. 압력/쿠션/배럴2-5온도차/사출-충전시간차잔차4개를기존흔들림4개에추가해총8개입력. RPM을새관계입력에사용하지않음.',
'RF/ExtraTrees/CatBoost3개를개별및동일평균결합하고기존RF2신호와도평균/max결합. 모델설정/결합가중치는사전고정. 기준은학습내부구간제외정상점수상위0.5/1/2%로정하고전체보고. 평가오탐률보장아님.',
'RH정상566가스17전체유지,미성형제외. 과거평가정상427가스13. 원본센서이력확보조건이며실제Run이아닌관측단절구간. 동일자료반복탐색의후보로독립최종성능아님. F1공식대회지표는확인필요.',
'','| 평가 | 모델/결합 | AP | TP | FP | FN | TN | F1 |','|---|---|---:|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} |')
(root/'reports/rg3_unlabeled_relation_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print('EXTERNAL POOL',len(u),u.TimeStamp.min(),u.TimeStamp.max())
print(summary.to_string(index=False))


             scheme              model  normal_budget  shots  gas       AP  TP  FP  FN  TN       F1
leave_one_block_out        RF_baseline           0.01    583   17 0.168625   4   6  13 560 0.296296
leave_one_block_out old_augmented_mean           0.01    583   17 0.098458   3  13  14 553 0.181818
leave_one_block_out  old_augmented_max           0.01    583   17 0.156458   2  10  15 556 0.137931
leave_one_block_out      augmented_Cat           0.01    583   17 0.069589   0   8  17 558 0.000000
leave_one_block_out    augmented3_mean           0.01    583   17 0.065088   0   9  17 557 0.000000
leave_one_block_out       augmented_ET           0.01    583   17 0.063722   0   3  17 563 0.000000
leave_one_block_out       augmented_RF           0.01    583   17 0.050807   0  13  17 553 0.000000
   past_blocks_only        RF_baseline           0.01    440   13 0.081203   2   4  11 423 0.210526
   past_blocks_only old_augmented_mean           0.01    440   13 0.085304   2   6  11 421 0.190476
